# ToS;DR LLM Evaluation Pipeline
This is where I will test out evaluating LLMs on ToS;DR dataset. After I get this right, I will merge this notebook with `100_tos_evaluation.ipynb` into one big evaluation pipeline.

## TODO: 
- prepare the evaluation pipeline (its ok if it'lll have its own format for now)
- run the LLM evaluation pipeline.
- - i don't have a rubric for bad, good, or neutral. I'll need help from some LLM to do that.


### Problems
- ~21,000 are rows are too many for my Ollama daily usage limits.
- Can I instead run a function to do it like, every 1000 rows and then save the results into their own dataframes and then export to csv and then move on to the next batch? 
    - Either that or i just run it on one loop but I save it every X times.
    - basta i wanna never lose my progress at any point. always save it and i can start from where I left off last time.

In [44]:
# Trying to make tos_dr json files into dataframe.
import pandas as pd
import json

In [45]:
JSON_PATH = "../../generated_files/tos_dr"
SERVICES_JSON_PATH = f"{JSON_PATH}/tosdr_services_denormalized.json"
TOSDR_FILES_PATH = "../../generated_files/tos_dr"
TOPICS_SCORED_JSON_PATH = "../../generated_files/tosdr_topics_scored.json"

In [46]:
# Load the Amazon JSON file
with open(f"{JSON_PATH}/amazon_service.json", 'r') as f:
    data = json.load(f)

# Create a DataFrame from the JSON data
df_amazon = pd.json_normalize(
    data,
    record_path=['cases', 'points'],
    meta=[
        'service_name',
        ['cases', 'case_id'],
        ['cases', 'case_classification'],
        ['cases', 'case_title'],
        ['cases', 'case_description'],
        ['cases', 'case_topic'],        
    ]
)

df_amazon.head()

,point_id,point_title,point_source,point_analysis,point_quote_text,point_quote_start,point_quote_end,point_document_id,service_name,cases.case_id,cases.case_classification,cases.case_title,cases.case_description,cases.case_topic
0,1582,personal data is given to third parties,https://www.amazon.com/gp/help/customer/displa...,Amazon may release your data when they believe...,This includes exchanging information with othe...,8238.0,8361.0,38.0,Amazon,188,bad,This service gives your personal data to third...,Your personal data is or may be given to third...,Third Parties
1,5929,This service can share your personal informati...,https://www.amazon.com/gp/help/customer/displa...,Generated through the annotate view,The Amazon Group Companies are subject to the ...,4616.0,4952.0,1051.0,Amazon,188,bad,This service gives your personal data to third...,Your personal data is or may be given to third...,Third Parties
2,1122,The service uses your personal data for advert...,https://www.amazon.com/gp/help/customer/displa...,Amazon uses your personal data and your behavi...,"To serve you interest-based ads, we use inform...",525.0,643.0,39.0,Amazon,216,bad,Your personal data is used for advertising,Your interaction with the service and data you...,Advertising
3,5925,This service employs separate policies for dif...,https://www.amazon.com/gp/help/customer/displa...,Generated through the annotate view,"Please review our other policies, such as our ...",16304.0,16441.0,37.0,Amazon,200,neutral,Separate policies are employed for different p...,The user may need to read additional product-s...,Transparency
4,5923,This service forces users into binding arbitra...,https://www.amazon.com/gp/help/customer/displa...,Generated through the annotate view,Any dispute or claim relating in any way to yo...,13750.0,14060.0,37.0,Amazon,339,bad,You are forced into binding arbitration in cas...,This service forces users to use their own con...,Dispute Resolution


## Ingest all ToS;DR points

One row per point across every service in `tosdr_services_denormalized.json`. Services (or cases) without points are skipped.

In [47]:
POINT_META = [
    "service_name",
    ["cases", "case_id"],
    ["cases", "case_classification"],
    ["cases", "case_title"],
    ["cases", "case_description"],
    ["cases", "case_topic"],
]


def load_tosdr_points_df(json_path: str) -> pd.DataFrame:
    """Load all ToS;DR points from denormalized services JSON.

    Returns one row per point with the same columns as the single-service
    `pd.json_normalize(..., record_path=['cases', 'points'])` approach.
    Services without cases/points are skipped.
    """
    with open(json_path, "r") as f:
        services = json.load(f)

    services_with_points = []
    for service in services:
        cases_with_points = [
            case for case in service.get("cases", []) if case.get("points")
        ]
        if cases_with_points:
            services_with_points.append({**service, "cases": cases_with_points})

    if not services_with_points:
        return pd.DataFrame()

    return pd.json_normalize(
        services_with_points,
        record_path=["cases", "points"],
        meta=POINT_META,
    )

In [48]:
def format_topic_rubric(scores: list[dict]) -> str:
    """Format a topic's score descriptions into a multi-line rubric."""
    return "\n".join(
        f"{item['score']}: {item['description']}"
        for item in sorted(scores, key=lambda x: x["score"])
    )


def load_topic_rubrics(json_path: str) -> dict[str, str]:
    """Load topic title -> formatted score_rubric mapping."""
    with open(json_path, "r") as f:
        topics = json.load(f)
    return {
        topic["title"]: format_topic_rubric(topic["scores"])
        for topic in topics
    }


def attach_score_rubric(
    points_df: pd.DataFrame,
    topics_json_path: str,
    topic_col: str = "cases.case_topic",
) -> pd.DataFrame:
    """Add score_rubric column by matching each row's case topic."""
    rubrics = load_topic_rubrics(topics_json_path)
    out = points_df.copy()
    out["score_rubric"] = out[topic_col].map(rubrics)
    return out

In [49]:
tos_points_df = attach_score_rubric(
    load_tosdr_points_df(SERVICES_JSON_PATH),
    TOPICS_SCORED_JSON_PATH,
)

In [50]:
EXCLUDED_TOPICS = {"[Deprecated]", "Unclassified"}

tos_points_df = (
    tos_points_df[~tos_points_df["cases.case_topic"].isin(EXCLUDED_TOPICS)]
    .reset_index(drop=True)
)

print(f"Points after excluding deprecated/unclassified topics: {len(tos_points_df):,}")

Points after excluding deprecated/unclassified topics: 23,868


In [51]:
tos_points_df.head()

,point_id,point_title,point_source,point_analysis,point_quote_text,point_quote_start,point_quote_end,point_document_id,service_name,cases.case_id,cases.case_classification,cases.case_title,cases.case_description,cases.case_topic,score_rubric
0,17470,The service provides information about how the...,https://telegram.org/privacy,Generated through the annotate view,<li>what we may use your personal data for;,3690.0,3734.0,2058.0,Telegram,227,good,Information is provided about how your persona...,The Privacy Policy explains the purposes for w...,Transparency,"-1: The terms and policies are inaccessible, u..."
1,17474,The service does not use third-party analytics...,https://telegram.org/privacy,Generated through the annotate view,We do not use cookies for profiling or adverti...,10113.0,10165.0,2058.0,Telegram,381,good,No third-party analytics or tracking platforms...,There are no Google Analytics or other trackin...,Third Parties,"-1: Personal data is shared with, or sold to, ..."
2,17480,You can delete your content from this service,https://telegram.org/privacy,Generated through the annotate view,"Deleting your account removes all messages, me...",22518.0,22723.0,2058.0,Telegram,175,good,You can delete your content from this service,You can ask the service to remove your content...,Right to Leave The Service,"-1: The user cannot freely terminate, or canno..."
3,8186,This service is only available to users of a c...,https://telegram.org/tos,None,Citizens of EU countries and the United Kingdo...,402.0,495.0,2059.0,Telegram,152,neutral,This service is only available to users over a...,The Services are intended for users who are at...,Governance,-1: The company retains wholly unilateral and ...
4,8188,This service does not sell your personal data,https://telegram.org/faq,None,"We don’t use your data for ad targeting, we do...",10819.0,10886.0,2060.0,Telegram,193,good,Your personal data is not sold,This service makes an explicit promise not to ...,Personal Data,"-1: Once collected, the user has no control ov..."


In [52]:
tos_points_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23868 entries, 0 to 23867
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   point_id                   23868 non-null  int64  
 1   point_title                23868 non-null  object 
 2   point_source               23753 non-null  object 
 3   point_analysis             23774 non-null  object 
 4   point_quote_text           22405 non-null  object 
 5   point_quote_start          22175 non-null  float64
 6   point_quote_end            22175 non-null  float64
 7   point_document_id          22398 non-null  float64
 8   service_name               23868 non-null  object 
 9   cases.case_id              23868 non-null  object 
 10  cases.case_classification  23868 non-null  object 
 11  cases.case_title           23868 non-null  object 
 12  cases.case_description     21631 non-null  object 
 13  cases.case_topic           23868 non-null  obj

In [53]:
print(f"Total points: {len(tos_points_df):,}")
print(f"Services: {tos_points_df['service_name'].nunique():,}")
print(f"Cases: {tos_points_df['cases.case_id'].nunique():,}")
print(f"\nClassifications:\n{tos_points_df['cases.case_classification'].value_counts()}")

Total points: 23,868
Services: 1,896
Cases: 243

Classifications:
cases.case_classification
neutral    10192
bad         6939
good        6248
blocker      489
Name: count, dtype: int64


In [33]:
# (Optional) Save dataframe to csv
tos_points_df.to_csv(f"{TOSDR_FILES_PATH}/tos_dr_points.csv", index=False)

## ToS;DR LLM Evaluation Pipeline

Prepare point-level rows for LLM evaluation, then run the same strict JSON scoring pipeline used in the 100-ToS notebook.

In [54]:
import os
import re
import html
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from sklearn.metrics import f1_score

load_dotenv()

OLLAMA_API_KEY = os.environ.get("OLLAMA_API_KEY")

if not OLLAMA_API_KEY:
    print("OLLAMA_API_KEY is still not set. Check .env path/value, then re-run this cell.")

candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
repo_root = None
for root in candidate_roots:
    if (root / "generated_files/tos_dr").exists():
        repo_root = root
        break

if repo_root is None:
    raise FileNotFoundError("Could not resolve repository root containing generated_files/tos_dr.")

print(f"Resolved repository root: {repo_root}")

Resolved repository root: /Users/riki/Coding Projects/Thesis/lawgic


In [55]:
GROUND_TRUTH_SCORE_MAP = {
    "bad": -1,
    "neutral": 0,
    "good": 1,
}


def clean_referenced_text(text: str) -> str:
    """Remove simple HTML/formatting noise and normalize whitespace."""
    if pd.isna(text):
        return ""
    cleaned = html.unescape(str(text))
    cleaned = re.sub(r"<[^>]+>", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned.strip()


tos_dr_eval_df = tos_points_df.copy()
tos_dr_eval_df["Referenced_Text"] = tos_dr_eval_df["point_quote_text"].apply(clean_referenced_text)
tos_dr_eval_df["Ground_Truth_Score"] = tos_dr_eval_df["cases.case_classification"].map(GROUND_TRUTH_SCORE_MAP)

tos_dr_eval_df = (
    tos_dr_eval_df[
        tos_dr_eval_df["Ground_Truth_Score"].isin([-1, 0, 1])
        & tos_dr_eval_df["Referenced_Text"].ne("")
        & tos_dr_eval_df["score_rubric"].notna()
    ]
    .assign(
        Company=lambda df: df["service_name"],
        Classification_Code=lambda df: df["cases.case_topic"],
        Specific_Rubric=lambda df: df["score_rubric"],
    )
    .reset_index(drop=True)
)

print(f"Rows in tos_points_df: {len(tos_points_df):,}")
print(f"Rows in ToS;DR eval dataframe: {len(tos_dr_eval_df):,}")
print(f"\nGround truth scores:\n{tos_dr_eval_df['Ground_Truth_Score'].value_counts().sort_index()}")
tos_dr_eval_df.head()

Rows in tos_points_df: 23,868
Rows in ToS;DR eval dataframe: 21,947

Ground truth scores:
Ground_Truth_Score
-1.0    6547
 0.0    9697
 1.0    5703
Name: count, dtype: int64


,point_id,point_title,point_source,point_analysis,point_quote_text,point_quote_start,point_quote_end,point_document_id,service_name,cases.case_id,cases.case_classification,cases.case_title,cases.case_description,cases.case_topic,score_rubric,Referenced_Text,Ground_Truth_Score,Company,Classification_Code,Specific_Rubric
0,17470,The service provides information about how the...,https://telegram.org/privacy,Generated through the annotate view,<li>what we may use your personal data for;,3690.0,3734.0,2058.0,Telegram,227,good,Information is provided about how your persona...,The Privacy Policy explains the purposes for w...,Transparency,"-1: The terms and policies are inaccessible, u...",what we may use your personal data for;,1.0,Telegram,Transparency,"-1: The terms and policies are inaccessible, u..."
1,17474,The service does not use third-party analytics...,https://telegram.org/privacy,Generated through the annotate view,We do not use cookies for profiling or adverti...,10113.0,10165.0,2058.0,Telegram,381,good,No third-party analytics or tracking platforms...,There are no Google Analytics or other trackin...,Third Parties,"-1: Personal data is shared with, or sold to, ...",We do not use cookies for profiling or adverti...,1.0,Telegram,Third Parties,"-1: Personal data is shared with, or sold to, ..."
2,17480,You can delete your content from this service,https://telegram.org/privacy,Generated through the annotate view,"Deleting your account removes all messages, me...",22518.0,22723.0,2058.0,Telegram,175,good,You can delete your content from this service,You can ask the service to remove your content...,Right to Leave The Service,"-1: The user cannot freely terminate, or canno...","Deleting your account removes all messages, me...",1.0,Telegram,Right to Leave The Service,"-1: The user cannot freely terminate, or canno..."
3,8186,This service is only available to users of a c...,https://telegram.org/tos,None,Citizens of EU countries and the United Kingdo...,402.0,495.0,2059.0,Telegram,152,neutral,This service is only available to users over a...,The Services are intended for users who are at...,Governance,-1: The company retains wholly unilateral and ...,Citizens of EU countries and the United Kingdo...,0.0,Telegram,Governance,-1: The company retains wholly unilateral and ...
4,8188,This service does not sell your personal data,https://telegram.org/faq,None,"We don’t use your data for ad targeting, we do...",10819.0,10886.0,2060.0,Telegram,193,good,Your personal data is not sold,This service makes an explicit promise not to ...,Personal Data,"-1: Once collected, the user has no control ov...","We don’t use your data for ad targeting, we do...",1.0,Telegram,Personal Data,"-1: Once collected, the user has no control ov..."


### Prepare Output and Prompt for LLMs

In [56]:
class OrdinalScoreOutput(BaseModel):
    score: Literal[-1, 0, 1] = Field(
        description="Ordinal severity score. Must be exactly one of -1, 0, or 1."
    )

In [57]:
prompt = PromptTemplate.from_template(
    """
You are a strict evaluator for Terms of Service clauses.
Assign exactly one ordinal score: -1, 0, or 1.
Use only provided specific rubric for this classification.
Do not explain reasoning.

Company: {Company}
Classification_Code: {Classification_Code}
Clause:
{Referenced_Text}

Specific_Rubric:
{Specific_Rubric}

Return strict JSON only in this exact shape:
{{"score": -1}}
or
{{"score": 0}}
or
{{"score": 1}}
""".strip()
)

### Define Metrics and Evaluation Runner

In [58]:
def _recover_score_from_parse_error(exc: Exception):
    """Recover -1/0/1 when model returns text like `score: 1` instead of JSON."""
    text = str(exc)
    if "Invalid json output:" not in text:
        return None

    first_line = text.splitlines()[0]
    raw_output = first_line.split("Invalid json output:", 1)[-1].strip()
    match = re.search(r"(?<!\d)(-1|0|1)(?!\d)", raw_output)
    if not match:
        return None
    return int(match.group(1))


def run_llm_evaluation(num_rows: int | None = None, output_filename: str = "tos_dr_eval.csv"):
    """
    Run evaluation on tos_dr_eval_df.

    Args:
        num_rows: Number of rows to evaluate. If None, evaluate all rows.
        output_filename: Output CSV filename in generated_files/tos_dr.

    Returns:
        final_eval_df, macro_f1_or_none
    """
    if num_rows is None:
        eval_df = tos_dr_eval_df.copy()
    else:
        if num_rows <= 0:
            raise ValueError("num_rows must be a positive integer or None.")
        eval_df = tos_dr_eval_df.head(num_rows).copy()

    total_rows = len(eval_df)
    print(f"Starting inference for {total_rows} rows...")

    predicted_scores = []
    for row_number, (idx, row) in enumerate(eval_df.iterrows(), start=1):
        payload = {
            "Company": row["Company"],
            "Classification_Code": row["Classification_Code"],
            "Referenced_Text": row["Referenced_Text"],
            "Specific_Rubric": row["Specific_Rubric"],
        }

        predicted_score = None
        try:
            response = evaluation_chain.invoke(payload)
            predicted_score = int(response.score)
        except Exception as exc:
            recovered_score = _recover_score_from_parse_error(exc)
            if recovered_score in (-1, 0, 1):
                predicted_score = recovered_score
                print(
                    f"[{row_number}/{total_rows}] parser recovery used -> {predicted_score}"
                )
            else:
                print(f"[{row_number}/{total_rows}] ERROR ({type(exc).__name__}): {exc}")

        predicted_scores.append(predicted_score)
        print(
            f"[{row_number}/{total_rows}] "
            f"{row['Company']} | {row['Classification_Code']} -> {predicted_score}"
        )

    final_eval_df = eval_df.copy()
    final_eval_df["Predicted_Score"] = predicted_scores

    output_path = repo_root / "generated_files/tos_dr" / output_filename
    output_path.parent.mkdir(parents=True, exist_ok=True)
    final_eval_df.to_csv(output_path, index=False)
    print(f"Saved evaluated rows to: {output_path}")

    valid_mask = (
        final_eval_df["Ground_Truth_Score"].isin([-1, 0, 1])
        & final_eval_df["Predicted_Score"].isin([-1, 0, 1])
    )

    macro_f1 = None
    print(f"Rows included in report: {len(final_eval_df)}")
    if valid_mask.any():
        macro_f1 = f1_score(
            final_eval_df.loc[valid_mask, "Ground_Truth_Score"].astype(int),
            final_eval_df.loc[valid_mask, "Predicted_Score"].astype(int),
            average="macro",
        )
        print(f"Macro F1 (valid predictions only): {macro_f1:.4f}")
        print(f"Valid rows used for F1: {int(valid_mask.sum())}/{len(final_eval_df)}")
    else:
        print("No valid predictions available yet for F1 calculation.")

    return final_eval_df, macro_f1

### Batch Evaluation (Ollama Daily Limit)

Full-dataset run (~21k rows) can exceed Ollama Cloud daily quota. Use `run_llm_eval_by_batch(start_row, end_row)` to evaluate a slice of `tos_dr_eval_df`, save progress to CSV after each batch, and combine batch files later.

Row indices are **0-based** and **half-open**: `start_row` is inclusive, `end_row` is exclusive. Example: `(0, 5000)` evaluates rows 0–4999.

In [59]:
def run_llm_eval_by_batch(
    start_row: int,
    end_row: int,
    output_filename: str | None = None,
):
    """
    Run LLM evaluation on a slice of tos_dr_eval_df.

    Args:
        start_row: Inclusive 0-based start index.
        end_row: Exclusive 0-based end index.
        output_filename: Optional CSV filename in generated_files/tos_dr.
            Defaults to tos_dr_eval_gemma4_31b_rows_{start}_{end}.csv

    Returns:
        final_eval_df, macro_f1_or_none
    """
    if start_row < 0 or end_row <= start_row:
        raise ValueError("Require 0 <= start_row < end_row.")
    if end_row > len(tos_dr_eval_df):
        raise ValueError(
            f"end_row {end_row} exceeds dataframe length {len(tos_dr_eval_df)}."
        )

    eval_df = tos_dr_eval_df.iloc[start_row:end_row].copy()
    total_rows = len(eval_df)

    if output_filename is None:
        output_filename = f"tos_dr_eval_gemma4_31b_rows_{start_row}_{end_row}.csv"

    print(
        f"Starting inference for rows {start_row}:{end_row} "
        f"({total_rows} rows)..."
    )

    predicted_scores = []
    for row_number, (idx, row) in enumerate(eval_df.iterrows(), start=1):
        payload = {
            "Company": row["Company"],
            "Classification_Code": row["Classification_Code"],
            "Referenced_Text": row["Referenced_Text"],
            "Specific_Rubric": row["Specific_Rubric"],
        }

        predicted_score = None
        try:
            response = evaluation_chain.invoke(payload)
            predicted_score = int(response.score)
        except Exception as exc:
            recovered_score = _recover_score_from_parse_error(exc)
            if recovered_score in (-1, 0, 1):
                predicted_score = recovered_score
                print(
                    f"[{row_number}/{total_rows}] parser recovery used -> {predicted_score}"
                )
            else:
                print(f"[{row_number}/{total_rows}] ERROR ({type(exc).__name__}): {exc}")

        predicted_scores.append(predicted_score)
        print(
            f"[{row_number}/{total_rows}] "
            f"{row['Company']} | {row['Classification_Code']} -> {predicted_score}"
        )

    final_eval_df = eval_df.copy()
    final_eval_df["Predicted_Score"] = predicted_scores

    output_path = repo_root / "generated_files/tos_dr" / output_filename
    output_path.parent.mkdir(parents=True, exist_ok=True)
    final_eval_df.to_csv(output_path, index=False)
    print(f"Saved evaluated rows to: {output_path}")

    valid_mask = (
        final_eval_df["Ground_Truth_Score"].isin([-1, 0, 1])
        & final_eval_df["Predicted_Score"].isin([-1, 0, 1])
    )

    macro_f1 = None
    print(f"Rows included in report: {len(final_eval_df)}")
    if valid_mask.any():
        macro_f1 = f1_score(
            final_eval_df.loc[valid_mask, "Ground_Truth_Score"].astype(int),
            final_eval_df.loc[valid_mask, "Predicted_Score"].astype(int),
            average="macro",
        )
        print(f"Macro F1 (valid predictions only): {macro_f1:.4f}")
        print(f"Valid rows used for F1: {int(valid_mask.sum())}/{len(final_eval_df)}")
    else:
        print("No valid predictions available yet for F1 calculation.")

    return final_eval_df, macro_f1

### Gemma4:31B via Ollama Cloud

In [60]:
llm = ChatOllama(
    model="gemma4:31b",
    base_url="https://ollama.com",
    temperature=0.0,
    format="json",
    client_kwargs={
        "headers": {
            "Authorization": f"Bearer {OLLAMA_API_KEY}",
        }
    },
)

structured_llm = llm.with_structured_output(OrdinalScoreOutput)

evaluation_chain = prompt | structured_llm
print("LangChain + Ollama Cloud pipeline ready (model: gemma4:31b, format=json).")

LangChain + Ollama Cloud pipeline ready (model: gemma4:31b, format=json).


### Test Evaluation Run

In [41]:
# Quick test run: set num_rows to any positive integer.
quick_eval_df, quick_macro_f1 = run_llm_evaluation(
    num_rows=10,
    output_filename="tos_dr_eval_gemma4_31b_quick.csv",
)
quick_eval_df.head()

Starting inference for 10 rows...
[1/10] Telegram | Transparency -> 0
[2/10] Telegram | Third Parties -> 1
[3/10] Telegram | Right to Leave The Service -> -1
[4/10] Telegram | Governance -> 0
[5/10] Telegram | Personal Data -> 1
[6/10] Telegram | Logs -> 1
[7/10] Telegram | Notice of Changing Terms -> -1
[8/10] Telegram | Anonymity -> 1
[9/10] Telegram | Transparency -> 1
[10/10] Telegram | Security -> 0
Saved evaluated rows to: /Users/riki/Coding Projects/Thesis/lawgic/generated_files/tos_dr/tos_dr_eval_gemma4_31b_quick.csv
Rows included in report: 10
Macro F1 (valid predictions only): 0.5778
Valid rows used for F1: 10/10


,point_id,point_title,point_source,point_analysis,point_quote_text,point_quote_start,point_quote_end,point_document_id,service_name,cases.case_id,...,cases.case_title,cases.case_description,cases.case_topic,score_rubric,Referenced_Text,Ground_Truth_Score,Company,Classification_Code,Specific_Rubric,Predicted_Score
0,17470,The service provides information about how the...,https://telegram.org/privacy,Generated through the annotate view,<li>what we may use your personal data for;,3690.0,3734.0,2058.0,Telegram,227,...,Information is provided about how your persona...,The Privacy Policy explains the purposes for w...,Transparency,"-1: The terms and policies are inaccessible, u...",what we may use your personal data for;,1.0,Telegram,Transparency,"-1: The terms and policies are inaccessible, u...",0
1,17474,The service does not use third-party analytics...,https://telegram.org/privacy,Generated through the annotate view,We do not use cookies for profiling or adverti...,10113.0,10165.0,2058.0,Telegram,381,...,No third-party analytics or tracking platforms...,There are no Google Analytics or other trackin...,Third Parties,"-1: Personal data is shared with, or sold to, ...",We do not use cookies for profiling or adverti...,1.0,Telegram,Third Parties,"-1: Personal data is shared with, or sold to, ...",1
2,17480,You can delete your content from this service,https://telegram.org/privacy,Generated through the annotate view,"Deleting your account removes all messages, me...",22518.0,22723.0,2058.0,Telegram,175,...,You can delete your content from this service,You can ask the service to remove your content...,Right to Leave The Service,"-1: The user cannot freely terminate, or canno...","Deleting your account removes all messages, me...",1.0,Telegram,Right to Leave The Service,"-1: The user cannot freely terminate, or canno...",-1
3,8186,This service is only available to users of a c...,https://telegram.org/tos,None,Citizens of EU countries and the United Kingdo...,402.0,495.0,2059.0,Telegram,152,...,This service is only available to users over a...,The Services are intended for users who are at...,Governance,-1: The company retains wholly unilateral and ...,Citizens of EU countries and the United Kingdo...,0.0,Telegram,Governance,-1: The company retains wholly unilateral and ...,0
4,8188,This service does not sell your personal data,https://telegram.org/faq,None,"We don’t use your data for ad targeting, we do...",10819.0,10886.0,2060.0,Telegram,193,...,Your personal data is not sold,This service makes an explicit promise not to ...,Personal Data,"-1: Once collected, the user has no control ov...","We don’t use your data for ad targeting, we do...",1.0,Telegram,Personal Data,"-1: Once collected, the user has no control ov...",1


### Full Evaluation Run

Problem: there are too many points. I am hitting my Ollama daily session limit lol...

In [ ]:
# Full run (disabled): use batch cells below instead to stay within Ollama daily limits.
# full_eval_df, full_macro_f1 = run_llm_evaluation(
#     output_filename="tos_dr_eval_gemma4_31b_full.csv",
# )
# full_eval_df.head()

### Batch Evaluation Runs (5000 rows each)

Run one batch cell per session. Each batch saves its own CSV under `generated_files/tos_dr/`. Combine batch CSVs after all batches finish.

In [61]:
BATCH_SIZE = 5000

total_eval_rows = len(tos_dr_eval_df)
batch_ranges = [
    (start, min(start + BATCH_SIZE, total_eval_rows))
    for start in range(0, total_eval_rows, BATCH_SIZE)
]

print(f"Total eval rows: {total_eval_rows:,}")
print(f"Batch size: {BATCH_SIZE:,}")
print(f"Number of batches: {len(batch_ranges)}")
for batch_number, (start_row, end_row) in enumerate(batch_ranges, start=1):
    print(
        f"Batch {batch_number}: rows {start_row}:{end_row} "
        f"({end_row - start_row:,} rows) -> "
        f"tos_dr_eval_gemma4_31b_rows_{start_row}_{end_row}.csv"
    )

Total eval rows: 21,947
Batch size: 5,000
Number of batches: 5
Batch 1: rows 0:5000 (5,000 rows) -> tos_dr_eval_gemma4_31b_rows_0_5000.csv
Batch 2: rows 5000:10000 (5,000 rows) -> tos_dr_eval_gemma4_31b_rows_5000_10000.csv
Batch 3: rows 10000:15000 (5,000 rows) -> tos_dr_eval_gemma4_31b_rows_10000_15000.csv
Batch 4: rows 15000:20000 (5,000 rows) -> tos_dr_eval_gemma4_31b_rows_15000_20000.csv
Batch 5: rows 20000:21947 (1,947 rows) -> tos_dr_eval_gemma4_31b_rows_20000_21947.csv


In [62]:
# Batch 1/5: rows 0-4999
batch_1_df, batch_1_macro_f1 = run_llm_eval_by_batch(0, 5000)
batch_1_df.head()

Starting inference for rows 0:5000 (5000 rows)...
[1/5000] Telegram | Transparency -> 0
[2/5000] Telegram | Third Parties -> 1
[3/5000] Telegram | Right to Leave The Service -> -1
[4/5000] Telegram | Governance -> 0
[5/5000] Telegram | Personal Data -> 0
[6/5000] Telegram | Logs -> 0
[7/5000] Telegram | Notice of Changing Terms -> -1
[8/5000] Telegram | Anonymity -> 1
[9/5000] Telegram | Transparency -> 1
[10/5000] Telegram | Security -> 0
[11/5000] Telegram | Trackers -> 0
[12/5000] Telegram | Types of Information Collected -> 1
[13/5000] Telegram | Right to Leave The Service -> 1
[14/5000] Telegram | Content -> 0
[15/5000] Telegram | Transparency -> 1
[16/5000] Telegram | Transparency -> 0
[17/5000] Telegram | Governance -> 0
[18/5000] Telegram | Content -> 1
[19/5000] Telegram | Suspension and Censorship -> 0
[20/5000] Telegram | User Choice -> 1
[21/5000] Telegram | Transparency -> -1
[22/5000] Telegram | Law and Government Requests -> 0
[23/5000] Telegram | Security -> 0
[24/5000]

,point_id,point_title,point_source,point_analysis,point_quote_text,point_quote_start,point_quote_end,point_document_id,service_name,cases.case_id,...,cases.case_title,cases.case_description,cases.case_topic,score_rubric,Referenced_Text,Ground_Truth_Score,Company,Classification_Code,Specific_Rubric,Predicted_Score
0,17470,The service provides information about how the...,https://telegram.org/privacy,Generated through the annotate view,<li>what we may use your personal data for;,3690.0,3734.0,2058.0,Telegram,227,...,Information is provided about how your persona...,The Privacy Policy explains the purposes for w...,Transparency,"-1: The terms and policies are inaccessible, u...",what we may use your personal data for;,1.0,Telegram,Transparency,"-1: The terms and policies are inaccessible, u...",0
1,17474,The service does not use third-party analytics...,https://telegram.org/privacy,Generated through the annotate view,We do not use cookies for profiling or adverti...,10113.0,10165.0,2058.0,Telegram,381,...,No third-party analytics or tracking platforms...,There are no Google Analytics or other trackin...,Third Parties,"-1: Personal data is shared with, or sold to, ...",We do not use cookies for profiling or adverti...,1.0,Telegram,Third Parties,"-1: Personal data is shared with, or sold to, ...",1
2,17480,You can delete your content from this service,https://telegram.org/privacy,Generated through the annotate view,"Deleting your account removes all messages, me...",22518.0,22723.0,2058.0,Telegram,175,...,You can delete your content from this service,You can ask the service to remove your content...,Right to Leave The Service,"-1: The user cannot freely terminate, or canno...","Deleting your account removes all messages, me...",1.0,Telegram,Right to Leave The Service,"-1: The user cannot freely terminate, or canno...",-1
3,8186,This service is only available to users of a c...,https://telegram.org/tos,None,Citizens of EU countries and the United Kingdo...,402.0,495.0,2059.0,Telegram,152,...,This service is only available to users over a...,The Services are intended for users who are at...,Governance,-1: The company retains wholly unilateral and ...,Citizens of EU countries and the United Kingdo...,0.0,Telegram,Governance,-1: The company retains wholly unilateral and ...,0
4,8188,This service does not sell your personal data,https://telegram.org/faq,None,"We don’t use your data for ad targeting, we do...",10819.0,10886.0,2060.0,Telegram,193,...,Your personal data is not sold,This service makes an explicit promise not to ...,Personal Data,"-1: Once collected, the user has no control ov...","We don’t use your data for ad targeting, we do...",1.0,Telegram,Personal Data,"-1: Once collected, the user has no control ov...",0


In [ ]:
# Batch 2/5: rows 5000-9999
batch_2_df, batch_2_macro_f1 = run_llm_eval_by_batch(5000, 10000)
batch_2_df.head()

In [ ]:
# Batch 3/5: rows 10000-14999
batch_3_df, batch_3_macro_f1 = run_llm_eval_by_batch(10000, 15000)
batch_3_df.head()

In [ ]:
# Batch 4/5: rows 15000-19999
batch_4_df, batch_4_macro_f1 = run_llm_eval_by_batch(15000, 20000)
batch_4_df.head()

In [ ]:
# Batch 5/5: rows 20000-end (final partial batch)
batch_5_df, batch_5_macro_f1 = run_llm_eval_by_batch(20000, len(tos_dr_eval_df))
batch_5_df.head()

In [ ]:
# After all batches finish, combine batch CSVs into one dataframe.
# import glob
#
# batch_csv_paths = sorted(glob.glob(str(repo_root / "generated_files/tos_dr/tos_dr_eval_gemma4_31b_rows_*.csv")))
# combined_eval_df = pd.concat((pd.read_csv(path) for path in batch_csv_paths), ignore_index=True)
# combined_eval_df.to_csv(repo_root / "generated_files/tos_dr/tos_dr_eval_gemma4_31b_full.csv", index=False)
# print(f"Combined rows: {len(combined_eval_df):,}")

## Batch Eval v2: Multi-Row Comparison Study

**Purpose:** Test whether evaluating multiple ToS points in one LLM call saves Ollama quota without hurting accuracy.

**Baseline:** First 100 rows from `tos_dr_eval_gemma4_31b_rows_0_5000.csv` (single-row `gemma4:31b` predictions).

**Method:** Batch Eval v2 groups rows by `Classification_Code` (same topic/rubric), sends multiple clauses per call, and returns strict JSON:

```json
{"results": [{"row_id": 17470, "score": 1}, {"row_id": 17474, "score": 0}]}
```

**Comparison runs:** batch sizes **5**, **10**, and **20** on the same 100 rows.

**Metrics reported per batch size:**
- Macro F1 vs `Ground_Truth_Score`
- Agreement with baseline `Predicted_Score`
- Number of LLM calls (batch calls + any single-row fallbacks)
- Parse/missing prediction counts

In [63]:
class BatchScoreItem(BaseModel):
    row_id: int = Field(description="point_id from the input clause list")
    score: Literal[-1, 0, 1] = Field(
        description="Ordinal severity score. Must be exactly one of -1, 0, or 1."
    )


class BatchScoreOutput(BaseModel):
    results: list[BatchScoreItem] = Field(
        description="One score per input row_id. Every input row_id must appear exactly once."
    )


batch_prompt = PromptTemplate.from_template(
    """
You are a strict evaluator for Terms of Service clauses.
Assign exactly one ordinal score: -1, 0, or 1 for EACH clause below.
Use only the provided specific rubric for this classification.
Do not explain reasoning.

Classification_Code: {Classification_Code}

Specific_Rubric:
{Specific_Rubric}

Clauses:
{Clauses_JSON}

Return strict JSON only in this exact shape:
{{"results": [{{"row_id": 123, "score": -1}}, {{"row_id": 456, "score": 0}}]}}

Each input row_id must appear exactly once in results.
""".strip()
)

batch_structured_llm = llm.with_structured_output(BatchScoreOutput)
batch_evaluation_chain = batch_prompt | batch_structured_llm

BASELINE_EVAL_PATH = repo_root / "generated_files/tos_dr/tos_dr_eval_gemma4_31b_rows_0_5000.csv"
COMPARISON_NUM_ROWS = 100
COMPARISON_BATCH_SIZES = [5, 10, 20]

baseline_single_row_df = pd.read_csv(BASELINE_EVAL_PATH).head(COMPARISON_NUM_ROWS).copy()
baseline_single_row_df["point_id"] = baseline_single_row_df["point_id"].astype(int)

print(f"Loaded baseline rows: {len(baseline_single_row_df)}")
print(f"Baseline macro F1: ", end="")
_baseline_valid = baseline_single_row_df["Predicted_Score"].isin([-1, 0, 1])
print(
    f"{f1_score(baseline_single_row_df.loc[_baseline_valid, 'Ground_Truth_Score'].astype(int), baseline_single_row_df.loc[_baseline_valid, 'Predicted_Score'].astype(int), average='macro'):.4f}"
)
baseline_single_row_df.head()

Loaded baseline rows: 100
Baseline macro F1: 0.4288


,point_id,point_title,point_source,point_analysis,point_quote_text,point_quote_start,point_quote_end,point_document_id,service_name,cases.case_id,...,cases.case_title,cases.case_description,cases.case_topic,score_rubric,Referenced_Text,Ground_Truth_Score,Company,Classification_Code,Specific_Rubric,Predicted_Score
0,17470,The service provides information about how the...,https://telegram.org/privacy,Generated through the annotate view,<li>what we may use your personal data for;,3690.0,3734.0,2058.0,Telegram,227,...,Information is provided about how your persona...,The Privacy Policy explains the purposes for w...,Transparency,"-1: The terms and policies are inaccessible, u...",what we may use your personal data for;,1.0,Telegram,Transparency,"-1: The terms and policies are inaccessible, u...",0
1,17474,The service does not use third-party analytics...,https://telegram.org/privacy,Generated through the annotate view,We do not use cookies for profiling or adverti...,10113.0,10165.0,2058.0,Telegram,381,...,No third-party analytics or tracking platforms...,There are no Google Analytics or other trackin...,Third Parties,"-1: Personal data is shared with, or sold to, ...",We do not use cookies for profiling or adverti...,1.0,Telegram,Third Parties,"-1: Personal data is shared with, or sold to, ...",1
2,17480,You can delete your content from this service,https://telegram.org/privacy,Generated through the annotate view,"Deleting your account removes all messages, me...",22518.0,22723.0,2058.0,Telegram,175,...,You can delete your content from this service,You can ask the service to remove your content...,Right to Leave The Service,"-1: The user cannot freely terminate, or canno...","Deleting your account removes all messages, me...",1.0,Telegram,Right to Leave The Service,"-1: The user cannot freely terminate, or canno...",-1
3,8186,This service is only available to users of a c...,https://telegram.org/tos,NaN,Citizens of EU countries and the United Kingdo...,402.0,495.0,2059.0,Telegram,152,...,This service is only available to users over a...,The Services are intended for users who are at...,Governance,-1: The company retains wholly unilateral and ...,Citizens of EU countries and the United Kingdo...,0.0,Telegram,Governance,-1: The company retains wholly unilateral and ...,0
4,8188,This service does not sell your personal data,https://telegram.org/faq,NaN,"We don’t use your data for ad targeting, we do...",10819.0,10886.0,2060.0,Telegram,193,...,Your personal data is not sold,This service makes an explicit promise not to ...,Personal Data,"-1: Once collected, the user has no control ov...","We don’t use your data for ad targeting, we do...",1.0,Telegram,Personal Data,"-1: Once collected, the user has no control ov...",0


In [64]:
def build_topic_batches(df: pd.DataFrame, batch_size: int) -> list[pd.DataFrame]:
    """Group by topic/rubric, then chunk into fixed-size batches."""
    batches = []
    for _, topic_df in df.groupby("Classification_Code", sort=False):
        topic_df = topic_df.reset_index(drop=True)
        for start in range(0, len(topic_df), batch_size):
            batches.append(topic_df.iloc[start : start + batch_size].copy())
    return batches


def _format_clauses_json(batch_df: pd.DataFrame) -> str:
    items = [
        {
            "row_id": int(row["point_id"]),
            "company": row["Company"],
            "clause": row["Referenced_Text"],
        }
        for _, row in batch_df.iterrows()
    ]
    return json.dumps(items, ensure_ascii=False, indent=2)


def _predict_single_row(row: pd.Series) -> int | None:
    payload = {
        "Company": row["Company"],
        "Classification_Code": row["Classification_Code"],
        "Referenced_Text": row["Referenced_Text"],
        "Specific_Rubric": row["Specific_Rubric"],
    }
    try:
        response = evaluation_chain.invoke(payload)
        return int(response.score)
    except Exception as exc:
        recovered_score = _recover_score_from_parse_error(exc)
        if recovered_score in (-1, 0, 1):
            return recovered_score
        print(
            f"Single-row fallback failed for point_id={row['point_id']} "
            f"({type(exc).__name__}): {exc}"
        )
        return None


def run_batch_eval_v2(
    eval_df: pd.DataFrame,
    batch_size: int,
    fallback_to_single: bool = True,
) -> tuple[pd.DataFrame, dict]:
    """Evaluate multiple rows per LLM call, grouped by topic/rubric."""
    batches = build_topic_batches(eval_df, batch_size)
    predicted_by_point_id: dict[int, int | None] = {}
    stats = {
        "batch_size": batch_size,
        "num_rows": len(eval_df),
        "num_batches": len(batches),
        "batch_calls": 0,
        "fallback_calls": 0,
        "batch_parse_failures": 0,
        "missing_predictions": 0,
    }

    for batch_number, batch_df in enumerate(batches, start=1):
        topic = batch_df["Classification_Code"].iloc[0]
        expected_ids = [int(x) for x in batch_df["point_id"]]
        batch_scores: dict[int, int | None] = {}

        payload = {
            "Classification_Code": topic,
            "Specific_Rubric": batch_df["Specific_Rubric"].iloc[0],
            "Clauses_JSON": _format_clauses_json(batch_df),
        }

        try:
            response = batch_evaluation_chain.invoke(payload)
            stats["batch_calls"] += 1
            for item in response.results:
                batch_scores[int(item.row_id)] = int(item.score)
        except Exception as exc:
            stats["batch_parse_failures"] += 1
            print(
                f"Batch {batch_number}/{len(batches)} "
                f"({topic}, {len(batch_df)} rows) ERROR ({type(exc).__name__}): {exc}"
            )

        for point_id in expected_ids:
            score = batch_scores.get(point_id)
            if score in (-1, 0, 1):
                predicted_by_point_id[point_id] = score
                continue

            if fallback_to_single:
                row = batch_df.loc[batch_df["point_id"] == point_id].iloc[0]
                fallback_score = _predict_single_row(row)
                stats["fallback_calls"] += 1
                predicted_by_point_id[point_id] = fallback_score
            else:
                predicted_by_point_id[point_id] = None

        print(
            f"Batch {batch_number}/{len(batches)} | {topic} | "
            f"{len(batch_df)} rows | resolved={sum(predicted_by_point_id[pid] in (-1, 0, 1) for pid in expected_ids)}/{len(expected_ids)}"
        )

    out = eval_df.copy()
    out["Predicted_Score_Batch"] = out["point_id"].map(predicted_by_point_id)
    stats["missing_predictions"] = int(out["Predicted_Score_Batch"].isna().sum())
    stats["total_llm_calls"] = stats["batch_calls"] + stats["fallback_calls"]
    return out, stats

In [65]:
def summarize_batch_v2_comparison(
    baseline_df: pd.DataFrame,
    batch_df: pd.DataFrame,
    stats: dict,
) -> dict:
    """Compare batch-v2 predictions against baseline single-row predictions."""
    merged = baseline_df.merge(
        batch_df[["point_id", "Predicted_Score_Batch"]],
        on="point_id",
        how="left",
    )

    baseline_valid = merged["Predicted_Score"].isin([-1, 0, 1])
    batch_valid = merged["Predicted_Score_Batch"].isin([-1, 0, 1])
    both_valid = baseline_valid & batch_valid

    summary = {
        "batch_size": stats["batch_size"],
        "num_rows": stats["num_rows"],
        "num_batches": stats["num_batches"],
        "batch_calls": stats["batch_calls"],
        "fallback_calls": stats["fallback_calls"],
        "total_llm_calls": stats["total_llm_calls"],
        "batch_parse_failures": stats["batch_parse_failures"],
        "missing_predictions": stats["missing_predictions"],
        "baseline_macro_f1": None,
        "batch_macro_f1": None,
        "agreement_with_baseline": None,
        "agreement_rows_compared": int(both_valid.sum()),
    }

    if baseline_valid.any():
        summary["baseline_macro_f1"] = f1_score(
            merged.loc[baseline_valid, "Ground_Truth_Score"].astype(int),
            merged.loc[baseline_valid, "Predicted_Score"].astype(int),
            average="macro",
        )

    if batch_valid.any():
        summary["batch_macro_f1"] = f1_score(
            merged.loc[batch_valid, "Ground_Truth_Score"].astype(int),
            merged.loc[batch_valid, "Predicted_Score_Batch"].astype(int),
            average="macro",
        )

    if both_valid.any():
        summary["agreement_with_baseline"] = (
            merged.loc[both_valid, "Predicted_Score"].astype(int)
            == merged.loc[both_valid, "Predicted_Score_Batch"].astype(int)
        ).mean()

    return summary


def run_batch_v2_comparison_study(
    baseline_df: pd.DataFrame,
    batch_sizes: list[int],
    output_prefix: str = "tos_dr_batch_v2_compare",
) -> tuple[pd.DataFrame, dict[int, pd.DataFrame]]:
    """Run batch-v2 for multiple batch sizes and collect comparison summaries."""
    summaries = []
    batch_results_by_size: dict[int, pd.DataFrame] = {}

    for batch_size in batch_sizes:
        print("\n" + "=" * 72)
        print(f"Running Batch Eval v2 | batch_size={batch_size}")
        print("=" * 72)

        batch_result_df, stats = run_batch_eval_v2(
            baseline_df,
            batch_size=batch_size,
            fallback_to_single=True,
        )
        summary = summarize_batch_v2_comparison(baseline_df, batch_result_df, stats)
        summaries.append(summary)
        batch_results_by_size[batch_size] = batch_result_df

        output_path = (
            repo_root
            / "generated_files/tos_dr"
            / f"{output_prefix}_batch{batch_size}_rows{len(baseline_df)}.csv"
        )
        batch_result_df.to_csv(output_path, index=False)
        print(f"Saved batch-v2 results to: {output_path}")

        print(
            f"batch_size={batch_size} | total_llm_calls={summary['total_llm_calls']} "
            f"| batch_macro_f1={summary['batch_macro_f1']:.4f} "
            f"| agreement_with_baseline={summary['agreement_with_baseline']:.4f}"
        )

    comparison_summary_df = pd.DataFrame(summaries)
    comparison_summary_path = (
        repo_root
        / "generated_files/tos_dr"
        / f"{output_prefix}_summary_rows{len(baseline_df)}.csv"
    )
    comparison_summary_df.to_csv(comparison_summary_path, index=False)
    print(f"\nSaved comparison summary to: {comparison_summary_path}")

    return comparison_summary_df, batch_results_by_size

### Run 100-Row Comparison (Batch Sizes 5, 10, 20)

Execute the cell below to run all three batch-size experiments on the same 100 baseline rows. Review:
- `batch_macro_f1` vs `baseline_macro_f1`
- `agreement_with_baseline` (how often batch-v2 matches your existing single-row predictions)
- `total_llm_calls` (quota efficiency)

In [66]:
batch_v2_comparison_summary_df, batch_v2_results_by_size = run_batch_v2_comparison_study(
    baseline_df=baseline_single_row_df,
    batch_sizes=COMPARISON_BATCH_SIZES,
    output_prefix="tos_dr_batch_v2_compare_gemma4_31b",
)

batch_v2_comparison_summary_df


Running Batch Eval v2 | batch_size=5
Batch 1/29 | Transparency | 5 rows | resolved=5/5
Batch 2/29 | Transparency | 5 rows | resolved=5/5
Batch 3/29 | Transparency | 5 rows | resolved=5/5
Batch 4/29 | Third Parties | 5 rows | resolved=5/5
Batch 5/29 | Third Parties | 1 rows | resolved=1/1
Batch 6/29 | Right to Leave The Service | 5 rows | resolved=5/5
Batch 7/29 | Governance | 5 rows | resolved=5/5
Batch 8/29 | Governance | 2 rows | resolved=2/2
Batch 9/29 | Personal Data | 5 rows | resolved=5/5
Batch 10/29 | Logs | 1 rows | resolved=1/1
Batch 11/29 | Notice of Changing Terms | 5 rows | resolved=5/5
Batch 12/29 | Notice of Changing Terms | 1 rows | resolved=1/1
Batch 13/29 | Anonymity | 2 rows | resolved=2/2
Batch 14/29 | Security | 4 rows | resolved=4/4
Batch 15/29 | Trackers | 5 rows | resolved=5/5
Batch 16/29 | Trackers | 5 rows | resolved=5/5
Batch 17/29 | Trackers | 5 rows | resolved=5/5
Batch 18/29 | Trackers | 2 rows | resolved=2/2
Batch 19/29 | Types of Information Collected | 

,batch_size,num_rows,num_batches,batch_calls,fallback_calls,total_llm_calls,batch_parse_failures,missing_predictions,baseline_macro_f1,batch_macro_f1,agreement_with_baseline,agreement_rows_compared
0,5,100,29,29,0,29,0,0,0.428788,0.458522,0.83,100
1,10,100,22,22,0,22,0,0,0.428788,0.459259,0.77,100
2,20,100,20,20,0,20,0,0,0.428788,0.459866,0.82,100


In [67]:
# Optional: inspect per-row disagreements for a chosen batch size.
INSPECT_BATCH_SIZE = 10

inspect_df = baseline_single_row_df.merge(
    batch_v2_results_by_size[INSPECT_BATCH_SIZE][["point_id", "Predicted_Score_Batch"]],
    on="point_id",
    how="left",
)
inspect_df["agrees_with_baseline"] = (
    inspect_df["Predicted_Score"].astype("Int64")
    == inspect_df["Predicted_Score_Batch"].astype("Int64")
)

disagreements_df = inspect_df[~inspect_df["agrees_with_baseline"]].copy()
print(f"Disagreements for batch_size={INSPECT_BATCH_SIZE}: {len(disagreements_df)}")
disagreements_df[
    [
        "point_id",
        "Company",
        "Classification_Code",
        "Ground_Truth_Score",
        "Predicted_Score",
        "Predicted_Score_Batch",
        "Referenced_Text",
    ]
].head(20)

Disagreements for batch_size=10: 23


,point_id,Company,Classification_Code,Ground_Truth_Score,Predicted_Score,Predicted_Score_Batch,Referenced_Text
2,17480,Telegram,Right to Leave The Service,1.0,-1,0,"Deleting your account removes all messages, me..."
5,8854,Telegram,Logs,1.0,0,1,"If collected, this metadata can be kept for 12..."
11,17477,Telegram,Types of Information Collected,0.0,1,0,"To improve the security of your account, as we..."
12,17479,Telegram,Right to Leave The Service,1.0,1,0,You can delete your Telegram account by procee...
13,17472,Telegram,Content,1.0,0,1,Secret chats use end-to-end encryption. This m...
18,8187,Telegram,Suspension and Censorship,0.0,0,1,i> Q: A bot or channel is infringing on my cop...
20,17469,Telegram,Transparency,0.0,-1,1,English Bahasa Indonesia Bahasa Melayu Deutsch...
21,17478,Telegram,Law and Government Requests,1.0,0,1,Telegram receives a court order that confirms ...
26,13423,RiseUp.net,Governance,0.0,-1,0,We are not liable for any damages related to t...
34,17345,RiseUp.net,Right to Leave The Service,1.0,1,0,You may choose to delete your riseup.net accou...


## Validation Slice: Rows 5000–6000 (Single-Row vs Batch v2)

Follow-up validation on an **unseen 1000-row slice** after the initial 100-row batch-v2 study.

**Plan:**
1. Run the normal single-row pipeline on rows `5000:6000` via `run_llm_eval_by_batch`.
2. Run Batch v2 on the same rows with `batch_size=20` (recommended setting).
3. Compare macro F1, agreement, and LLM call count.

Run the cells below in order when ready. Each step saves CSV output under `generated_files/tos_dr/`.

In [68]:
VALIDATION_START_ROW = 5000
VALIDATION_END_ROW = 6000
VALIDATION_BATCH_SIZE = 20

validation_slice_df = tos_dr_eval_df.iloc[VALIDATION_START_ROW:VALIDATION_END_ROW].copy()
validation_single_row_output = (
    f"tos_dr_eval_gemma4_31b_rows_{VALIDATION_START_ROW}_{VALIDATION_END_ROW}.csv"
)
validation_batch_v2_output = (
    f"tos_dr_eval_gemma4_31b_batchv2_batch{VALIDATION_BATCH_SIZE}_rows"
    f"{VALIDATION_START_ROW}_{VALIDATION_END_ROW}.csv"
)
validation_comparison_output = (
    f"tos_dr_eval_gemma4_31b_validation_{VALIDATION_START_ROW}_{VALIDATION_END_ROW}_summary.csv"
)

print(f"Validation slice rows: {len(validation_slice_df):,}")
print(f"Row range: {VALIDATION_START_ROW}:{VALIDATION_END_ROW}")
print(f"Batch v2 size: {VALIDATION_BATCH_SIZE}")
validation_slice_df.head()

Validation slice rows: 1,000
Row range: 5000:6000
Batch v2 size: 20


,point_id,point_title,point_source,point_analysis,point_quote_text,point_quote_start,point_quote_end,point_document_id,service_name,cases.case_id,cases.case_classification,cases.case_title,cases.case_description,cases.case_topic,score_rubric,Referenced_Text,Ground_Truth_Score,Company,Classification_Code,Specific_Rubric
5000,16330,"This service collects your IP address, which c...",https://support.guilded.gg/hc/en-us/articles/3...,Generated through the annotate view,"Whenever you interact with our Services, we au...",4363.0,4533.0,3806.0,Guilded,399,neutral,"Your IP address is collected, which can be use...",None,Types of Information Collected,-1: The categories of information collected ar...,"Whenever you interact with our Services, we au...",0.0,Guilded,Types of Information Collected,-1: The categories of information collected ar...
5001,16312,You have the right to leave this service at an...,https://support.guilded.gg/hc/en-us/articles/3...,Generated through the annotate view,What if I want to stop using the Services? </s...,33427.0,33564.0,3808.0,Guilded,170,good,You have the right to leave this service at an...,You can stop using the service and/or cancel o...,Right to Leave The Service,"-1: The user cannot freely terminate, or canno...",What if I want to stop using the Services? . Y...,1.0,Guilded,Right to Leave The Service,"-1: The user cannot freely terminate, or canno..."
5002,16313,The service can delete your account without pr...,https://support.guilded.gg/hc/en-us/articles/3...,Generated through the annotate view,Guilded is also free to terminate (or suspend ...,33745.0,33914.0,3808.0,Guilded,201,bad,Your account can be deleted without prior noti...,"At any time, your account can be terminated wi...",Suspension and Censorship,-1: The company reserves the right to suspend ...,Guilded is also free to terminate (or suspend ...,-1.0,Guilded,Suspension and Censorship,-1: The company reserves the right to suspend ...
5003,16320,Any liability on behalf of the service is only...,https://support.guilded.gg/hc/en-us/articles/3...,Generated through the annotate view,"(C) ANY AMOUNT, IN THE AGGREGATE, IN EXCESS OF...",41113.0,41409.0,3808.0,Guilded,149,bad,Any liability on behalf of the service is only...,None,Governance,-1: The company retains wholly unilateral and ...,"(C) ANY AMOUNT, IN THE AGGREGATE, IN EXCESS OF...",-1.0,Guilded,Governance,-1: The company retains wholly unilateral and ...
5004,16324,Failure to enforce any provision of the Terms ...,https://support.guilded.gg/hc/en-us/articles/3...,Generated through the annotate view,"The failure of either you or us to exercise, i...",25359.0,25494.0,3808.0,Guilded,295,neutral,Failure to enforce any provision of the Terms ...,Even if the service does not or not always enf...,Governance,-1: The company retains wholly unilateral and ...,"The failure of either you or us to exercise, i...",0.0,Guilded,Governance,-1: The company retains wholly unilateral and ...


### Step 1: Normal Single-Row Evaluation (Rows 5000–6000)

In [69]:
validation_single_row_df, validation_single_macro_f1 = run_llm_eval_by_batch(
    VALIDATION_START_ROW,
    VALIDATION_END_ROW,
    output_filename=validation_single_row_output,
)
validation_single_row_df.head()

Starting inference for rows 5000:6000 (1000 rows)...
[1/1000] Guilded | Types of Information Collected -> 1
[2/1000] Guilded | Right to Leave The Service -> 0
[3/1000] Guilded | Suspension and Censorship -> -1
[4/1000] Guilded | Governance -> 0
[5/1000] Guilded | Governance -> 0
[6/1000] Guilded | Trackers -> 0
[7/1000] Guilded | Personal Data -> -1
[8/1000] Guilded | Types of Information Collected -> 1
[9/1000] Guilded | Trackers -> 0
[10/1000] Guilded | Personal Data -> 0
[11/1000] Guilded | Transparency -> 1
[12/1000] Guilded | User Choice -> 1
[13/1000] Guilded | Transparency -> 1
[14/1000] Guilded | Transparency -> 1
[15/1000] Guilded | Third Parties -> 0
[16/1000] Guilded | Personal Data -> 0
[17/1000] Guilded | Right to Leave The Service -> 0
[18/1000] Guilded | Personal Data -> 0
[19/1000] Guilded | User Choice -> 1
[20/1000] Guilded | Right to Leave The Service -> 0
[21/1000] Guilded | Law and Government Requests -> 0
[22/1000] Guilded | Copyright License -> 1
[23/1000] Guilde

,point_id,point_title,point_source,point_analysis,point_quote_text,point_quote_start,point_quote_end,point_document_id,service_name,cases.case_id,...,cases.case_title,cases.case_description,cases.case_topic,score_rubric,Referenced_Text,Ground_Truth_Score,Company,Classification_Code,Specific_Rubric,Predicted_Score
5000,16330,"This service collects your IP address, which c...",https://support.guilded.gg/hc/en-us/articles/3...,Generated through the annotate view,"Whenever you interact with our Services, we au...",4363.0,4533.0,3806.0,Guilded,399,...,"Your IP address is collected, which can be use...",None,Types of Information Collected,-1: The categories of information collected ar...,"Whenever you interact with our Services, we au...",0.0,Guilded,Types of Information Collected,-1: The categories of information collected ar...,1.0
5001,16312,You have the right to leave this service at an...,https://support.guilded.gg/hc/en-us/articles/3...,Generated through the annotate view,What if I want to stop using the Services? </s...,33427.0,33564.0,3808.0,Guilded,170,...,You have the right to leave this service at an...,You can stop using the service and/or cancel o...,Right to Leave The Service,"-1: The user cannot freely terminate, or canno...",What if I want to stop using the Services? . Y...,1.0,Guilded,Right to Leave The Service,"-1: The user cannot freely terminate, or canno...",0.0
5002,16313,The service can delete your account without pr...,https://support.guilded.gg/hc/en-us/articles/3...,Generated through the annotate view,Guilded is also free to terminate (or suspend ...,33745.0,33914.0,3808.0,Guilded,201,...,Your account can be deleted without prior noti...,"At any time, your account can be terminated wi...",Suspension and Censorship,-1: The company reserves the right to suspend ...,Guilded is also free to terminate (or suspend ...,-1.0,Guilded,Suspension and Censorship,-1: The company reserves the right to suspend ...,-1.0
5003,16320,Any liability on behalf of the service is only...,https://support.guilded.gg/hc/en-us/articles/3...,Generated through the annotate view,"(C) ANY AMOUNT, IN THE AGGREGATE, IN EXCESS OF...",41113.0,41409.0,3808.0,Guilded,149,...,Any liability on behalf of the service is only...,None,Governance,-1: The company retains wholly unilateral and ...,"(C) ANY AMOUNT, IN THE AGGREGATE, IN EXCESS OF...",-1.0,Guilded,Governance,-1: The company retains wholly unilateral and ...,0.0
5004,16324,Failure to enforce any provision of the Terms ...,https://support.guilded.gg/hc/en-us/articles/3...,Generated through the annotate view,"The failure of either you or us to exercise, i...",25359.0,25494.0,3808.0,Guilded,295,...,Failure to enforce any provision of the Terms ...,Even if the service does not or not always enf...,Governance,-1: The company retains wholly unilateral and ...,"The failure of either you or us to exercise, i...",0.0,Guilded,Governance,-1: The company retains wholly unilateral and ...,0.0


### Step 2: Batch v2 Evaluation (Rows 5000–6000, batch_size=20)

In [70]:
validation_batch_v2_df, validation_batch_v2_stats = run_batch_eval_v2(
    validation_slice_df,
    batch_size=VALIDATION_BATCH_SIZE,
    fallback_to_single=True,
)

validation_batch_v2_path = repo_root / "generated_files/tos_dr" / validation_batch_v2_output
validation_batch_v2_df.to_csv(validation_batch_v2_path, index=False)
print(f"Saved batch-v2 validation results to: {validation_batch_v2_path}")
validation_batch_v2_df.head()

Batch 1/60 | Types of Information Collected | 20 rows | resolved=20/20
Batch 2/60 | Types of Information Collected | 2 rows | resolved=2/2
Batch 3/60 | Right to Leave The Service | 20 rows | resolved=20/20
Batch 4/60 | Right to Leave The Service | 20 rows | resolved=20/20
Batch 5/60 | Right to Leave The Service | 2 rows | resolved=2/2
Batch 6/60 | Suspension and Censorship | 20 rows | resolved=20/20
Batch 7/60 | Suspension and Censorship | 19 rows | resolved=19/19
Batch 8/60 | Governance | 20 rows | resolved=20/20
Batch 9/60 | Governance | 20 rows | resolved=20/20
Batch 10/60 | Governance | 20 rows | resolved=20/20
Batch 11/60 | Governance | 20 rows | resolved=20/20
Batch 12/60 | Governance | 18 rows | resolved=18/18
Batch 13/60 | Trackers | 20 rows | resolved=20/20
Batch 14/60 | Trackers | 20 rows | resolved=20/20
Batch 15/60 | Trackers | 20 rows | resolved=20/20
Batch 16/60 | Trackers | 20 rows | resolved=20/20
Batch 17/60 | Trackers | 20 rows | resolved=20/20
Batch 18/60 | Trackers 

,point_id,point_title,point_source,point_analysis,point_quote_text,point_quote_start,point_quote_end,point_document_id,service_name,cases.case_id,...,cases.case_title,cases.case_description,cases.case_topic,score_rubric,Referenced_Text,Ground_Truth_Score,Company,Classification_Code,Specific_Rubric,Predicted_Score_Batch
5000,16330,"This service collects your IP address, which c...",https://support.guilded.gg/hc/en-us/articles/3...,Generated through the annotate view,"Whenever you interact with our Services, we au...",4363.0,4533.0,3806.0,Guilded,399,...,"Your IP address is collected, which can be use...",None,Types of Information Collected,-1: The categories of information collected ar...,"Whenever you interact with our Services, we au...",0.0,Guilded,Types of Information Collected,-1: The categories of information collected ar...,1
5001,16312,You have the right to leave this service at an...,https://support.guilded.gg/hc/en-us/articles/3...,Generated through the annotate view,What if I want to stop using the Services? </s...,33427.0,33564.0,3808.0,Guilded,170,...,You have the right to leave this service at an...,You can stop using the service and/or cancel o...,Right to Leave The Service,"-1: The user cannot freely terminate, or canno...",What if I want to stop using the Services? . Y...,1.0,Guilded,Right to Leave The Service,"-1: The user cannot freely terminate, or canno...",1
5002,16313,The service can delete your account without pr...,https://support.guilded.gg/hc/en-us/articles/3...,Generated through the annotate view,Guilded is also free to terminate (or suspend ...,33745.0,33914.0,3808.0,Guilded,201,...,Your account can be deleted without prior noti...,"At any time, your account can be terminated wi...",Suspension and Censorship,-1: The company reserves the right to suspend ...,Guilded is also free to terminate (or suspend ...,-1.0,Guilded,Suspension and Censorship,-1: The company reserves the right to suspend ...,-1
5003,16320,Any liability on behalf of the service is only...,https://support.guilded.gg/hc/en-us/articles/3...,Generated through the annotate view,"(C) ANY AMOUNT, IN THE AGGREGATE, IN EXCESS OF...",41113.0,41409.0,3808.0,Guilded,149,...,Any liability on behalf of the service is only...,None,Governance,-1: The company retains wholly unilateral and ...,"(C) ANY AMOUNT, IN THE AGGREGATE, IN EXCESS OF...",-1.0,Guilded,Governance,-1: The company retains wholly unilateral and ...,0
5004,16324,Failure to enforce any provision of the Terms ...,https://support.guilded.gg/hc/en-us/articles/3...,Generated through the annotate view,"The failure of either you or us to exercise, i...",25359.0,25494.0,3808.0,Guilded,295,...,Failure to enforce any provision of the Terms ...,Even if the service does not or not always enf...,Governance,-1: The company retains wholly unilateral and ...,"The failure of either you or us to exercise, i...",0.0,Guilded,Governance,-1: The company retains wholly unilateral and ...,0


### Step 3: Compare Single-Row vs Batch v2 on Validation Slice

In [71]:
validation_comparison_summary = summarize_batch_v2_comparison(
    baseline_df=validation_single_row_df,
    batch_df=validation_batch_v2_df,
    stats=validation_batch_v2_stats,
)
validation_comparison_summary["validation_start_row"] = VALIDATION_START_ROW
validation_comparison_summary["validation_end_row"] = VALIDATION_END_ROW
validation_comparison_summary["single_row_calls"] = len(validation_single_row_df)
validation_comparison_summary["call_reduction_pct"] = (
    1
    - (
        validation_comparison_summary["total_llm_calls"]
        / validation_comparison_summary["single_row_calls"]
    )
)

validation_comparison_summary_df = pd.DataFrame([validation_comparison_summary])
validation_comparison_summary_path = (
    repo_root / "generated_files/tos_dr" / validation_comparison_output
)
validation_comparison_summary_df.to_csv(validation_comparison_summary_path, index=False)

print(f"Saved validation comparison summary to: {validation_comparison_summary_path}")
validation_comparison_summary_df

Saved validation comparison summary to: /Users/riki/Coding Projects/Thesis/lawgic/generated_files/tos_dr/tos_dr_eval_gemma4_31b_validation_5000_6000_summary.csv


,batch_size,num_rows,num_batches,batch_calls,fallback_calls,total_llm_calls,batch_parse_failures,missing_predictions,baseline_macro_f1,batch_macro_f1,agreement_with_baseline,agreement_rows_compared,validation_start_row,validation_end_row,single_row_calls,call_reduction_pct
0,20,1000,60,60,0,60,0,0,0.44389,0.468284,0.838839,999,5000,6000,1000,0.94


In [72]:
# Optional: inspect disagreements on the 5000-6000 validation slice.
validation_inspect_df = validation_single_row_df.merge(
    validation_batch_v2_df[["point_id", "Predicted_Score_Batch"]],
    on="point_id",
    how="left",
)
validation_inspect_df["agrees_with_single_row"] = (
    validation_inspect_df["Predicted_Score"].astype("Int64")
    == validation_inspect_df["Predicted_Score_Batch"].astype("Int64")
)

validation_disagreements_df = validation_inspect_df[
    ~validation_inspect_df["agrees_with_single_row"]
].copy()

print(
    f"Validation disagreements (batch_size={VALIDATION_BATCH_SIZE}): "
    f"{len(validation_disagreements_df):,} / {len(validation_inspect_df):,}"
)
validation_disagreements_df[
    [
        "point_id",
        "Company",
        "Classification_Code",
        "Ground_Truth_Score",
        "Predicted_Score",
        "Predicted_Score_Batch",
        "Referenced_Text",
    ]
].head(20)

Validation disagreements (batch_size=20): 161 / 1,000


,point_id,Company,Classification_Code,Ground_Truth_Score,Predicted_Score,Predicted_Score_Batch,Referenced_Text
1,16312,Guilded,Right to Leave The Service,1.0,0.0,1,What if I want to stop using the Services? . Y...
6,16338,Guilded,Personal Data,0.0,-1.0,0,Information That’s Been De-Identified: We may ...
7,16331,Guilded,Types of Information Collected,-1.0,1.0,0,geolocation data
23,16306,Guilded,Copyright License,-1.0,-1.0,0,You agree that the licenses you grant are roya...
31,16295,Guilded,Governance,0.0,-1.0,0,"ay after a change to the Terms is effective, t..."
35,16328,Guilded,Advertising,-1.0,-1.0,0,We may communicate with you if you’ve provided...
41,11755,Prezi,Governance,0.0,-1.0,0,If Prezi does not exercise or enforce any lega...
42,13844,Prezi,Anonymity,-1.0,-1.0,0,If you do choose to create an account and beco...
47,13891,Prezi,Transparency,0.0,1.0,0,Share login credentials or passwords or use of...
64,13849,Prezi,Trackers,1.0,1.0,0,Prezi cookies do not collect personal informat...
